# Assignment 1: Warmtenet network design

**Mathematical Optimization (XM_0051), VU Amsterdam, 2026-27.** Individual assignment, **due Monday 23 November 2026, 23:59** (Canvas). Late submissions are not accepted.

### How this assignment works
- **Personal instance.** The data are generated from your **student number**, so your numerical results, designs and answers differ from those of other students. Enter your number in Setup *before starting the questions*.
- **Questions.** There are four questions (Q1–Q4, 90 points) and a closing section *Verification and GenAI use* (10 points). Your grade is (number of points)/10. Q2, Q3 and Q4 use the Q1 model. Check it carefully in Q1c before proceeding.
- **What to submit.** Upload this notebook (`.ipynb`) to Canvas **with all outputs**, after running *Restart and run all*. Fill in the `ANSWERS` dictionary at the end. `check_answer_format` checks required keys and basic types only; `print_answer_block` prints the block that staff use for grading. Neither checks mathematical correctness or submits the notebook. Save it after running these cells and upload it yourself. Staff check the reported numbers and designs against your personal instance and mark the explanations by hand.
- **Course code.** You may adapt code from the course labs and worked solutions. Cite the exercise or cell and explain how you used it. You are responsible for the resulting code, results and explanations.
- **Runtime.** In our laptop calibration, individual reference solves took up to about 40 seconds. Allow additional time for setup and a complete notebook run. Timings vary with the instance and hardware, including on Colab. If a solve is unexpectedly slow, check the formulation size, variable bounds and solver status.
- **Generative AI** (VU Level 3) may be used in your work, provided that you explain its use and critically assess its output in the last section. You are responsible for every model, line of code, number and explanation you submit.

## Setup
Open the Canvas notebook in Google Colab and save a copy in your own Google Drive before editing. Run the Setup cells below. The first installs the assignment package and its dependencies; allow a minute or more on a new runtime. A1 needs no separate solver download.

On your own computer, use Python 3.13 and install the package in your notebook's environment with `python -m pip install "mo2026 @ git+https://github.com/alessandrozocca/MO2026@a1"`. Run the same cells in Jupyter. Repeat Setup whenever you start a new runtime.

In [ ]:
#@title Install assignment packages
import sys

if "google.colab" in sys.modules:
    %pip install -q "mo2026 @ git+https://github.com/alessandrozocca/MO2026@a1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

from mo2026.common import solve, relax_integrality, check_answer_format, print_answer_block
from mo2026.warmtenet import make_instance, one_pipe_example, plot_instance, plot_design

**Enter your student number** (digits only). Keep it unchanged throughout the assignment: all data depend on it.

In [ ]:
STUDENT_NUMBER = None  # e.g. 2712345 (use a text such as "2712345-1" only if the teachers ask you to)

assert STUDENT_NUMBER is not None, "Enter your student number first."

Use `solve(model)` for all solves in this assignment. It calls HiGHS, loads the solution into the Pyomo model and returns the status, objective value, solving time, number of branch-and-bound nodes and best bound in a dictionary. Run `help(solve)` for details.

## The story

The Netherlands aims to stop heating homes with natural gas. For dense urban districts, a **district-heating network** (*warmtenet*) is often the main alternative: heat from a waste incinerator, a data center or surface water is pumped through insulated pipes to thousands of homes. Construction costs are high. In 2024 Vattenfall announced that it no longer starts new large-scale heat-network projects in existing buildings, citing falling heat demand and rising costs ([Vattenfall](https://group.vattenfall.com/nl/newsroom/achtergrondartikel/2024/vattenfall-grootschalige-warmteprojecten-in-bestaande-bouw-niet-meer-opgestart)). Amsterdam has since shifted much of its plan toward individual heat pumps ([AT5](https://www.at5.nl/artikelen/230746/aardgasvrij-2040-niet-haalbaar-warmtepompen-amsterdam)), while municipalities work to establish public heat companies.

You work for **Grachtenwarmte**, a fictional public heat company, and are planning a network for the fictional district *Tweegrachten*. Its street grid is cut by two canals, which pipes may cross only at bridges. The board asks:

> **Which neighborhoods should be connected to a heat network, from which heat sources, and along which streets should the pipes run? The other neighborhoods will switch to individual heat pumps.**

Your personal instance is shown below, with neighborhoods, candidate heat sources and streets along which pipes may be built.

In [ ]:
inst = make_instance(STUDENT_NUMBER)
print(inst)
plot_instance(inst)
plt.show()

In [ ]:
display(inst.sources)
display(inst.neighborhoods)
inst.pipes.head()

### Data and units
Costs are **annualized**: investment is spread over the asset's lifetime, with operation and maintenance added. All costs are expressed in **k€ per year**. Heat is measured as **peak demand in MW** on the design day, i.e., the coldest day the network must be able to handle. The network, the costs and the demands are synthetic and illustrative. The objective measures the total modeled cost of heating the district, including the heat-pump alternative. It does not determine the heat company's profitability or household tariffs.

| Symbol | Python | Meaning |
|---|---|---|
| $V$ | `inst.V` | junctions (nodes) of the street network |
| $E$ | `inst.E` | candidate pipes: one undirected street segment $e=\{i,j\}$ each, stored as `(i, j)` with `i < j` |
| $A$ | `inst.A` | arcs: both directions $(i,j)$ and $(j,i)$ of every candidate pipe |
| $S \subseteq V$ | `inst.S` | candidate heat sources |
| $K \subseteq V$ | `inst.K` | neighborhoods |
| $f_e$ | `inst.pipe_cost[e]` | cost of building pipe $e$ (k€/yr) |
| $u_e$ | `inst.pipe_cap[e]` | capacity of pipe $e$ (MW, total over both directions) |
| $t_e$ | `inst.transport_cost[e]` | transport cost (pumping, heat losses) per MW sent through pipe $e$ (k€/yr per MW) |
| $F_s,\ C_s,\ c_s$ | `inst.open_cost[s]`, `inst.capacity[s]`, `inst.heat_cost[s]` | opening cost (k€/yr), capacity (MW) and heat cost (k€/yr per MW) of source $s$ |
| $d_k$ | `inst.demand[k]` | peak heat demand of neighborhood $k$ (MW, integer) |
| $h_k$ | `inst.hp_cost[k]` | cost of switching neighborhood $k$ to individual heat pumps (k€/yr) |

Use `inst.edge(i, j)` to get the pipe $e$ (as stored in `inst.E`) that contains arc $(i,j)$, and `inst.in_arcs[i]` and `inst.out_arcs[i]` for the lists of arcs entering and leaving node $i$. The model ignores heat losses, supply and return pipes, and hydraulics. A pipe may carry heat in either direction.

### The model
The decision variables are
- $y_e \in \{0,1\}$: 1 if pipe $e \in E$ is built;
- $o_s \in \{0,1\}$: 1 if source $s \in S$ is opened;
- $z_k \in \{0,1\}$: 1 if neighborhood $k \in K$ is connected to the network (if $z_k = 0$ it switches to heat pumps);
- $x_{ij} \ge 0$: heat flow (MW) on arc $(i,j) \in A$;
- $g_s \ge 0$: heat produced (MW) at source $s \in S$.

Grachtenwarmte minimizes the total annual cost

$$
\min \ \sum_{s\in S} \big(F_s\, o_s + c_s\, g_s\big) \;+\; \sum_{e\in E} f_e\, y_e \;+\; \sum_{(i,j)\in A} t_{e(i,j)}\, x_{ij} \;+\; \sum_{k\in K} h_k\,(1-z_k),
$$

where $e(i,j)$ denotes the pipe containing arc $(i,j)$. The source constraints $g_s \le C_s\, o_s$ for all $s \in S$ limit production to the capacity of an opened source. You will supply the remaining two constraint families in Q1.

---
## Q1. The base model (20 points)

**Q1a [6 pt].** Write the two missing constraint families in mathematical notation:
1. **flow balance** at every node $i \in V$: heat flowing in, minus heat flowing out, plus heat produced (if $i$ is a source) equals the heat delivered (if $i$ is a connected neighborhood);
2. **pipe linking**: heat can flow through pipe $e$ only if it is built, and at most $u_e$ MW in total over both directions.

Justify each family in one or two sentences. For the pipe constraint, explain why it bounds $x_{ij} + x_{ji}$ and not each direction separately.

*Write your answer here.*

**Q1b [7 pt].** The function `build_weak(inst)` below declares the variables, objective and source constraints before calling `add_q1_constraints(m, inst)`. Complete `add_q1_constraints` in the next cell using your Q1a constraints. Name them `m.balance` (indexed by node) and `m.pipe_link` (indexed by pipe). Do not change `build_weak` itself. Solve your instance and report the total cost, opened sources, connected neighborhoods and built pipes. Plot the design with `plot_design`.

*Hint:* use `inst.in_arcs[i]` and `inst.out_arcs[i]`.

In [ ]:
def build_weak(inst):
    m = pyo.ConcreteModel()
    m.y = pyo.Var(inst.E, domain=pyo.Binary)            # 1 if pipe e is built
    m.o = pyo.Var(inst.S, domain=pyo.Binary)            # 1 if source s is opened
    m.z = pyo.Var(inst.K, domain=pyo.Binary)            # 1 if neighborhood k is connected
    m.x = pyo.Var(inst.A, domain=pyo.NonNegativeReals)  # heat flow on arc (i, j), MW
    m.g = pyo.Var(inst.S, domain=pyo.NonNegativeReals)  # heat produced at source s, MW

    m.cost = pyo.Objective(expr=
        sum(inst.open_cost[s] * m.o[s] + inst.heat_cost[s] * m.g[s] for s in inst.S)
        + sum(inst.pipe_cost[e] * m.y[e] for e in inst.E)
        + sum(inst.transport_cost[inst.edge(i, j)] * m.x[i, j] for (i, j) in inst.A)
        + sum(inst.hp_cost[k] * (1 - m.z[k]) for k in inst.K))

    @m.Constraint(inst.S)
    def source_link(m, s):
        return m.g[s] <= inst.capacity[s] * m.o[s]

    add_q1_constraints(m, inst)
    return m

In [ ]:
def add_q1_constraints(m, inst):
    # Add your flow-balance and pipe-linking constraints to the model m here.
    pass

In [ ]:
# Write your (commented) code here.

**Q1c [7 pt] Verification.** Check your model independently.
1. A second solve repeats the same formulation. Write an **independent check**: a function `check_solution(inst, built, opened, connected, flows, production)` that uses only plain Python and the data, *not* your Pyomo model. Check every rule of the problem statement directly on the numerical solution: valid indices (pipes in $E$, sources in $S$, neighborhoods in $K$, a flow for every arc in $A$, production only at sources), finite, nonnegative flow and production values, flow balance at every node, no flow on pipes that are not built, pipe and source capacities, and no production at closed sources. Recompute the total cost from the data. Apply the checker to your Q1b solution and compare this cost with the solver's objective value.
2. Design **two sanity tests**: changes to the data whose effect on the optimal solution you can predict *without solving*. At least one of them must have an exactly known optimal value (for example, a change of units: express all heat quantities in kW instead of MW, scaling capacities, demands and per-MW costs consistently; or a change that makes the optimum obvious). State each prediction before running the test, then report whether the result agrees. Use `make_instance(STUDENT_NUMBER)` to get a fresh copy of the data before each change.

In [ ]:
# Write your (commented) code here.

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

---
## Q2. How strong is your formulation? (25 points)

Q1 represents heat for all neighborhoods by a single flow $x_{ij}$. Compare this **aggregated** (*weak*) formulation with a **disaggregated** (*strong*) formulation, examining both LP bounds and solver performance.

Computer lab W1 Exercise 6(a) provides an LP/root/MILP reporting scaffold and valid-inequality experiments; Exercise 6(c) compares aggregated and individual linking constraints. You may adapt that code here.

**Q2a [8 pt] One pipe, by hand.** Consider a one-pipe instance: one source that is already open (no opening cost, no heat cost), one neighborhood with demand $d$ and heat-pump cost $h$, and one pipe between them with cost $f$ and capacity $u \ge d$ (no transport cost). Your instance data appear below.
1. What is the optimal cost of the MILP?
2. Write the **LP relaxation** of the weak model of Q1 for this instance (replace $y, z \in \{0,1\}$ by $y, z \in [0,1]$). Compute its optimal value and an optimal solution $(y, z)$ by hand.
3. Replace the linking constraint $x \le u\,y$ by $x \le d\,y$. Explain why this is still a valid formulation of the MILP. Compute the optimal value of its LP relaxation.
4. Explain in your own words why the weak LP relaxation "pays" only a fraction of the pipe.

Report the three optimal values in `ANSWERS` without rounding (you may compute your formulas in Python).

In [ ]:
ex = one_pipe_example(STUDENT_NUMBER)
ex

*Write your answer here.*

In [ ]:
# Write your (commented) code here.

**Q2b [9 pt] The disaggregated formulation.** Give each neighborhood $k\in K$ its own commodity. Let $x^k_{ij}$ be the heat *for neighborhood $k$* on arc $(i,j)$, and $g^k_s$ the heat produced at source $s$ *for neighborhood $k$*. Write the full model: objective, flow balance per commodity, and the linking constraints

$$
x^k_{ij} + x^k_{ji} \le d_k\, y_e \quad \forall e=\{i,j\}\in E,\ k \in K, \qquad\qquad g^k_s \le d_k\, o_s \quad \forall s\in S,\ k\in K,
$$

together with the capacity constraints of pipes and sources on the *total* flow. Justify the formulation, implement it as `build_strong(inst)` (with variables `y, o, z` as before) and check that its optimal value equals that of Q1b.

*Hint:* you may use flow decomposition: a nonnegative flow splits into source-to-neighborhood path flows and cycles. Explain why cycles can be removed here and how the remaining paths can be assigned to neighborhoods. A proof of the decomposition theorem is not required.

*Write your answer here.*

In [ ]:
def build_strong(inst):
    m = pyo.ConcreteModel()
    # Your disaggregated model here (keep the names y, o, z for the binary variables).
    return m

**Q2c [8 pt] Experiments.** Compare both formulations on your main instance and on a larger one: `make_instance(STUDENT_NUMBER, size=s)` for `s` in `"main"` and `"large"`. Use one table to report the following for each instance and formulation:
- the number of variables and constraints (`m.nvariables()`, `m.nconstraints()`);
- the optimal value of the LP relaxation (use `relax_integrality` on a fresh model);
- the **root bound**: the best lower bound that HiGHS proves at the root node, before any branching. Obtain it with `solve(model, options={"mip_max_nodes": 1})["bound"]` on a fresh model. The status of such a run is usually `'other'`; `"bound"` is the proven lower bound, while `"objective"` (if present) is only the best solution found so far;
- the optimal MILP value;
- the **LP gap** $(\text{MILP} - \text{LP})/\text{MILP}$ and the **root gap** $(\text{MILP} - \text{root bound})/\text{MILP}$;
- the solving time and the number of branch-and-bound nodes of the full MILP solve.

Use *your* results to answer:
1. Which formulation has the stronger LP relaxation? Relate this to Q2a.
2. For the main instance, examine the weak LP solution. For each pipe with positive flow, compare $y_e$ with $(x_{ij}+x_{ji})/u_e$. Calculate the fraction of the total cost of these pipes paid by the LP. Is the LP's total pipe cost $\sum_e f_e\, y^{\text{LP}}_e$ below that of the optimal design in your instance? Does the fact that the LP value is below the MILP value imply this, component by component?
3. Is the strong formulation also faster on your instances? Compare the LP bound with the root bound, for both formulations, and inspect the root-node processing in HiGHS using the logging cell below. What would you expect from a solver that does not add cutting planes? State the conclusions your two instances support and their limitations.

**Solver log.** After completing Q1, run the provided logging cell below. It writes `q2c-root.log` in the working directory and prints it below the cell, so the log is retained even if the notebook does not capture native solver output. The cell removes the previous file because HiGHS otherwise appends to it.

For this item, one root-node log of the weak main-instance model is sufficient. Inspect `BestBound` and the `Cuts`/`InLp` columns; `BestSol` is the incumbent cost, not a lower bound. Use full MILP solves for the times and node counts in the table. Keep the printed log in the saved notebook outputs; no separate log-file submission is needed.

In [ ]:
# Write your (commented) code here.

In [ ]:
# Save and display a root-node log for the main instance.
from pathlib import Path

log_path = Path("q2c-root.log")
log_path.unlink(missing_ok=True)

root_report = solve(
    build_weak(inst),
    options={"mip_max_nodes": 1, "log_file": str(log_path)},
)
print(log_path.read_text())
print("Root bound:", root_report["bound"])

*Write your answer here.*

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

---
## Q3. Network structure and heat prices (20 points)

Fix the optimal design of Q1 (which pipes are built, which sources are open, which neighborhoods are connected). The remaining **linear** problem routes heat through the built network at minimum cost.

Computer lab W3 Exercise 3 treats fixed-design prices, signs, kinks and infeasible demand increases. Exercise 4(d) reviews the directed-incidence integrality theorem and its assumptions, which are used in Q3a. The W3 lab takes place on Friday 13 November; its code may be adapted here with attribution.

**Q3a [8 pt].** Build the Q1 model, fix all binary variables to their optimal values, and call `relax_integrality(m)` so that HiGHS treats the model as an LP (fixed binaries stay fixed). Solve it and check that the cost equals the Q1 optimum.
1. Check whether all flows $x_{ij}$ and production levels $g_s$ in the solution returned by HiGHS are integers.
2. Explain the structural reason for this integrality. The shared capacity constraint $x_{ij}+x_{ji} \le u_e$ requires care.
   - (a) Use $t_e > 0$ to show that no optimal solution sends heat in both directions of a pipe.
   - (b) Relate the optimal solutions to those of a directed min-cost-flow problem with *separate* bounds $x_{ij} \le u_e$ and $x_{ji} \le u_e$, in which the sources become ordinary arcs from a super-source.
   - (c) Identify the matrix, its relevant property, and the integral data that guarantee integral vertices for this min-cost-flow problem. Why is the solution returned by HiGHS (a vertex of your LP) then integral?

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

**Q3b [12 pt] Marginal peak-supply costs.** Add `m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)` to your LP from Q3a before solving. `m.dual[m.balance[k]]` then holds the dual value of the flow-balance constraint at node $k$; `m.dual[m.pipe_link[e]]` and `m.dual[m.source_link[s]]` hold the capacity duals.

1. Report the dual values of all *connected* neighborhoods. After correcting their sign if needed, which neighborhood $k^\ast$ has the highest marginal peak-supply cost?

2. Interpret this value, including its unit, and check the interpretation numerically. Keep the design fixed, change $d_{k^\ast}$ by $\pm\delta$ with $\delta = 0.1$ MW, re-solve, and compute the two finite differences

    $$
    \text{left} = \frac{\text{cost}(d_{k^\ast}) - \text{cost}(d_{k^\ast}-\delta)}{\delta}, \qquad \text{right} = \frac{\text{cost}(d_{k^\ast}+\delta) - \text{cost}(d_{k^\ast})}{\delta}.
    $$

    How do they relate to the dual value? Use these differences to check the *sign* of your duals: with the balance written as in Q1a, the dual is the change in cost per extra MW. If your duals have the opposite sign, explain why. If the increase makes the LP infeasible, explain why (and report `None` for the right difference in `ANSWERS`).

3. Explain the dual value of $k^\ast$ from the network: which source supplies the marginal MW, which transport costs are paid, and does any binding pipe or source capacity affect the price? Use the capacity duals to support your explanation.

4. Do these marginal prices suffice to decide whether a uniform household heat tariff creates cross-subsidies between neighborhoods? Explain which costs and consumption data would also be needed.

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

---
## Q4. Temperature level and a social-housing policy (25 points)

Extend the model to include the network temperature and a social-housing requirement.

Computer lab W2 Exercise 7 treats a binary product with the opposite objective sign and controlled tests of an incomplete extension. You may use it to prepare for Q4a and Q4c.

**Temperature level.** A heat network runs either at **high temperature** (HT, about 90 °C) or at **low temperature** (LT, about 50–70 °C). Introduce a binary variable $\tau$ with $\tau = 1$ for an LT network. Use the following stylized rules:
- in an LT network, every built pipe is cheaper: its cost becomes $(1-\theta) f_e$ with $\theta$ = `inst.lt_pipe_discount`;
- in an LT network, every **connected** neighborhood with older homes (`inst.old[k]` is `True`) needs a booster heat pump with cost $b_k$ = `inst.booster_cost[k]`;
- in an HT network, every **opened** low-temperature source (`inst.source_type[s] == "LT"`) needs a booster at the source with cost $B_s$ = `inst.source_booster_cost[s]`.

**Policy.** The municipality requires that at least a share $\rho$ = `inst.social_share_target` of all **social-housing homes** in the district (`inst.social_homes[k]`) is connected to the network.

**Q4a [10 pt].** Extend your model mathematically. Define all new variables, write all new constraints and the new objective, and explain each constraint. The cost of an LT pipe involves the product $y_e\,\tau$ of two binary variables: linearize it using the McCormick inequalities from Lecture 2. Explain which inequalities are needed at an optimum, given the sign with which $q_e$ appears in the objective.

*Write your answer here.*

**Q4b [8 pt].** Extend your Q1 (or Q2) model without copying it. Write a function that adds the new variables and constraints and updates the objective (`m.cost.expr = m.cost.expr + ...`).
1. Solve it for your instance. Report the cost, the temperature level, the boosters installed, and the connected neighborhoods, and plot the design.
2. The **price of the policy**: by how much does the optimal cost increase because of the social-housing requirement? Which neighborhoods differ between your optimal designs with and without the policy? (Optimal designs need not be unique.)
3. The **price of the other temperature level**: what is the optimal cost (with the policy) if $\tau$ is fixed to the other value? Explain the temperature choice using the cost components in your instance.

In [ ]:
def add_q4(m, inst, policy=True):
    # Add the Q4 variables and constraints to m and update m.cost.expr.
    return m

*Write your answer here.*

**Q4c [7 pt] Audit a chatbot model.** A colleague asked a chatbot to implement Q4 and received the function below. "It runs and the cost is even lower than yours," says the colleague. Find the **three errors** in the function **by testing**, as well as by inspection. For each error:
1. describe the error and its effect on the solutions;
2. give a test (a cell you run) that **fails** on the chatbot's version and **passes** on your corrected version;
3. give the corrected constraint(s).

Independently recompute the annual cost of the chatbot's design and compare it with the reported objective value. Make this comparison once for the whole design.

*Hint:* check every rule of the problem statement on the chatbot's solution. An error may not show up in its optimum on your instance (for example, a booster error is invisible if the chatbot chooses an HT network). In that case, force the situation, e.g. with `m.tau.fix(1)`: the test should expose the error regardless of the instance.

In [ ]:
def chatbot_q4(m, inst):
    old = [k for k in inst.K if inst.old[k]]
    lt_sources = [s for s in inst.S if inst.source_type[s] == "LT"]
    m.tau = pyo.Var(domain=pyo.Binary)          # 1 = low-temperature network
    m.w = pyo.Var(old, domain=pyo.Binary)       # booster at an old neighborhood
    m.v = pyo.Var(lt_sources, domain=pyo.Binary)  # booster at an LT source
    m.q = pyo.Var(inst.E, domain=pyo.Binary)    # q = y * tau
    # an LT network needs boosters at the old neighborhoods
    m.c1 = pyo.Constraint(old, rule=lambda m, k: m.w[k] >= m.tau - m.z[k])
    # an HT network needs boosters at the opened LT sources
    m.c2 = pyo.Constraint(lt_sources, rule=lambda m, s: m.v[s] >= m.o[s] - m.tau)
    # linearization of q = y * tau
    m.c3 = pyo.Constraint(inst.E, rule=lambda m, i, j: m.q[i, j] >= m.y[i, j] + m.tau - 1)
    # social-housing policy
    m.c4 = pyo.Constraint(expr=sum(inst.social_homes[k] * m.z[k] for k in inst.K)
                          >= inst.social_share_target * len(inst.K))
    m.cost.expr = (m.cost.expr
                   - sum(inst.lt_pipe_discount * inst.pipe_cost[e] * m.q[e] for e in inst.E)
                   + sum(inst.booster_cost[k] * m.w[k] for k in old)
                   + sum(inst.source_booster_cost[s] * m.v[s] for s in lt_sources))
    return m

In [ ]:
# Write your (commented) code here.

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

---
## Verification and GenAI use (10 points)

This section is graded mainly on the quality of your verification. Choosing not to use GenAI carries no penalty.

1. **Use of GenAI.** Did you use GenAI tools? If yes, state which tool(s), for which (sub)questions, and for what purpose, for example "generating a first version of the flow-balance constraints in Q1b" or "explaining an error message in Q3a". If no, say so.
2. **Verification.** Summarize your checks and refer to the relevant cells. For each check, state which errors it could detect and what it showed. Identify errors that the checks, taken together, could *not* detect. Correct execution is marked in the earlier questions; here we assess the coverage and interpretation of the verification as a whole. No additional tests are required. Examples: the independent check of Q1c, the finite differences in Q3b, the tests in Q4c, or tests of your own.
3. **One consequential correction or limitation.** Describe one case in which GenAI output, or your own first attempt, was wrong, incomplete or misleading: how you found out and what you changed. If nothing went wrong, describe instead an issue you investigated, or a limitation of your verification, and why it matters. Do not invent errors.

1. *...*
2. *...*
3. *...*

---
## Your answers

Enter **your own results** in the dictionary below. Lists of pipes are lists of pairs `[i, j]`. You may assign variables computed above, e.g. `"q1_cost": r1["objective"]`. Run `check_answer_format` to check required keys and basic types, then `print_answer_block` to print the block used by the staff grader. These functions do not check mathematical correctness or upload anything. Save the notebook with its outputs and upload the `.ipynb` file to Canvas.

In [ ]:
ANSWERS_TEMPLATE = {
    "student_number": "id",
    "q1_cost": "number", "q1_built_pipes": "list", "q1_open_sources": "list", "q1_connected": "list",
    "q2a_milp": "number", "q2a_weak_lp": "number", "q2a_strong_lp": "number",
    "q2c": "dict",  # {"main": {"weak_lp": .., "strong_lp": .., "milp": ..}, "large": {...}}
    "q3a_lp_cost": "number",
    "q3b_neighborhood": "number", "q3b_dual": "number", "q3b_left": "number", "q3b_right": "number_or_none",
    "q4_cost": "number", "q4_tau": "number", "q4_built_pipes": "list", "q4_open_sources": "list",
    "q4_connected": "list", "q4_policy_price": "number", "q4_other_tau_cost": "number",
}

In [ ]:
ANSWERS = {key: None for key in ANSWERS_TEMPLATE}
ANSWERS["student_number"] = STUDENT_NUMBER
# ANSWERS["q1_cost"] = ...

In [ ]:
check_answer_format(ANSWERS, ANSWERS_TEMPLATE)
print_answer_block(ANSWERS)